# LA 01 — Binance data readiness audit

Stage A, independent Liquidity Absorption branch. **Read-only sources; no signal, model, or P&L.** Probe days are selected by fixed chronology (first, midpoint, last), not outcomes. Outputs are written as CSV/JSON under `01/`.

**Timestamp fix (LA-A01-0.1.1):** Unix s/ms/us/ns and pandas datetime candle open times are normalized to UTC before audit joins. If this notebook was open before pulling the fix, restart the kernel and run all cells.

In [1]:
from pathlib import Path
import sys
import pandas as pd

p = Path.cwd().resolve()
while not (p / "src" / "data_engineering").exists():
    if p == p.parent:
        raise RuntimeError("Launch Jupyter from repository or subfolder")
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from src.data_engineering.modeling.liquidity_absorption_stage_a import VERSION, binance_root, run_audit
if VERSION == "LA-A01-0.1.0":
    raise RuntimeError("Old audit module loaded. Pull latest changes, restart kernel, then rerun.")
print("Repo:", p, "| Audit:", VERSION)

Repo: D:\Projects\deribit_market_data | Audit: LA-A01-0.1.1


In [2]:
DATA_ROOT = binance_root(p)
OUT = p / "notebooks" / "Liquidity_Absorption" / "01"
print("Binance root:", DATA_ROOT)
print("Output:", OUT)

Binance root: D:\MarketData\Binance
Output: D:\Projects\deribit_market_data\notebooks\Liquidity_Absorption\01


In [3]:
decision = run_audit(DATA_ROOT, OUT)
decision

{'module': 'LA-A01-0.1.1',
 'purpose': 'readiness and consistency audit only, not strategy validation',
 'symbols': ['BTCUSDT', 'ETHUSDT'],
 'binance_root': 'D:\\MarketData\\Binance',
 'ready_for_signal_design': True,
 'checks': {'all_present_sources_and_sample_integrity': True},
 'warning': 'Flow differences are descriptive; all-day aggTrades completeness is NOT proven by three-day probes. No edge is claimed.',
 'files': ['01_minute_quality.csv',
  '01_aggtrade_coverage.csv',
  '01_trade_probe_quality.csv',
  '01_trade_kline_flow_comparison.csv',
  '01_sample_volume_profile.csv']}

In [4]:
for filename in decision["files"]:
    print("\n==", filename)
    f = OUT / filename
    if f.exists() and f.stat().st_size:
        display(pd.read_csv(f).head(12))
    else:
        print("Empty file; data not available")


== 01_minute_quality.csv


,symbol,venue,status,rows,first_utc,last_utc,null_times,duplicate_minutes,out_of_order_minutes,missing_minutes_inside_span,non_one_minute_intervals,bad_price_rows,negative_volume_rows,taker_quote_exceeds_total_rows,file,missing_columns
0,BTCUSDT,perp,PRESENT,1067040.0,2024-10-01 00:00:00+00:00,2026-10-11 23:59:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D:\MarketData\Binance\BTCUSDT_um_1m.parquet,NaN
1,BTCUSDT,spot,BAD_SCHEMA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:\MarketData\Binance\basis\spot\BTCUSDT_spot_...,"high,low,open"
2,ETHUSDT,perp,PRESENT,1067040.0,2024-10-01 00:00:00+00:00,2026-10-11 23:59:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D:\MarketData\Binance\ETHUSDT_um_1m.parquet,NaN
3,ETHUSDT,spot,BAD_SCHEMA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:\MarketData\Binance\basis\spot\ETHUSDT_spot_...,"high,low,open"



== 01_aggtrade_coverage.csv


,symbol,daily_files,first_day,last_day,missing_days_within_span,missing_day_examples
0,BTCUSDT,92,2026-07-01,2026-09-30,0,NaN
1,ETHUSDT,92,2026-07-01,2026-09-30,0,NaN



== 01_trade_probe_quality.csv


,symbol,day,file,status,rows,invalid_trade_rows,null_aggressor_side,timestamp_descents,first_timestamp_utc,last_timestamp_utc,minute_count,profile_bin_count
0,BTCUSDT,2026-07-01,BTCUSDT-aggTrades-2026-07-01.parquet,PRESENT,1994155,0,0,0,2026-07-01 00:00:00.058000+00:00,2026-07-01 23:59:59.987000+00:00,1440,300
1,BTCUSDT,2026-08-16,BTCUSDT-aggTrades-2026-08-16.parquet,PRESENT,280210,0,0,0,2026-08-16 00:00:00.002000+00:00,2026-08-16 23:59:59.807000+00:00,1440,56
2,BTCUSDT,2026-09-30,BTCUSDT-aggTrades-2026-09-30.parquet,PRESENT,1434005,0,0,0,2026-09-30 00:00:00.261000+00:00,2026-09-30 23:59:59.591000+00:00,1440,163
3,ETHUSDT,2026-07-01,ETHUSDT-aggTrades-2026-07-01.parquet,PRESENT,1380099,0,0,0,2026-07-01 00:00:00.034000+00:00,2026-07-01 23:59:59.849000+00:00,1440,294
4,ETHUSDT,2026-08-16,ETHUSDT-aggTrades-2026-08-16.parquet,PRESENT,232453,0,0,0,2026-08-16 00:00:00.003000+00:00,2026-08-16 23:59:59.518000+00:00,1440,65
5,ETHUSDT,2026-09-30,ETHUSDT-aggTrades-2026-09-30.parquet,PRESENT,1109197,0,0,0,2026-09-30 00:00:00.003000+00:00,2026-09-30 23:59:59.683000+00:00,1440,152



== 01_trade_kline_flow_comparison.csv


,symbol,day,matched_minutes,duplicate_kline_minutes,status
0,BTCUSDT,2026-07-01,0,1353,DUPLICATE_KLINE_MINUTES
1,BTCUSDT,2026-08-16,0,1353,DUPLICATE_KLINE_MINUTES
2,BTCUSDT,2026-09-30,0,1353,DUPLICATE_KLINE_MINUTES
3,ETHUSDT,2026-07-01,0,1353,DUPLICATE_KLINE_MINUTES
4,ETHUSDT,2026-08-16,0,1353,DUPLICATE_KLINE_MINUTES
5,ETHUSDT,2026-09-30,0,1353,DUPLICATE_KLINE_MINUTES



== 01_sample_volume_profile.csv


,symbol,day,log_price_bin,quote,trade_rows
0,BTCUSDT,2026-07-01,55082,5.028349e+08,11810
1,BTCUSDT,2026-07-01,55016,1.982021e+08,13376
2,BTCUSDT,2026-07-01,54923,1.758154e+08,15329
3,BTCUSDT,2026-07-01,54931,1.588320e+08,11292
4,BTCUSDT,2026-07-01,55007,1.537560e+08,14181
5,BTCUSDT,2026-07-01,54906,1.472137e+08,24780
6,BTCUSDT,2026-07-01,54924,1.378266e+08,17084
7,BTCUSDT,2026-07-01,54926,1.358090e+08,16931
8,BTCUSDT,2026-07-01,54902,1.357879e+08,21501
9,BTCUSDT,2026-07-01,55028,1.352497e+08,17964


## Interpretation checklist

- Are BTC and ETH perpetual minute bars present, monotone and free of duplicate open times?
- How many missing one-minute intervals occur within their historical spans?
- Are Binance spot 1m data available for the same assets?
- Are aggTrade files contiguous between the first and last day?
- Are sampled individual trades timestamp-ordered with valid price, quantity, aggressor side?
- Is the price/flow reconciliation broadly plausible? **This is descriptive; no statistical gate has been pre-registered for the relative error.**
- Do not interpret a top-volume bin as a predictive support/resistance zone. This is merely an audit visualization.

Commit `01/` outputs and the executed notebook to GitHub. Next: specify LA 02 ex-ante event definitions before screening outcomes.